# Compile results for Imiomics and LSS paper

In [ ]:
%matplotlib ipympl 

from pathlib import Path
from itertools import product
import string
from collections import defaultdict
import datetime

import SimpleITK as sitk
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import ListedColormap
from mpl_toolkits.axes_grid1 import make_axes_locatable
from cmap import Colormap

from spinemira.core.io import load_image, load_label_map
from spinemira.io.bids import Layout
from spinemira.core.segmentation.labels import TotalSpineSegLabels

from common.common import ImageBundle

In [ ]:
# Save file configuration
DPI = 300
SAVE_FILES = True
IMAGE_FILE_FORMATS_TO_EXPORT = [".eps", ".tiff", ".png", ".pdf"]
OUTPUT_FOLDER = Path("/workspace/data/figures") / datetime.date.today().strftime(
    "%Y-%m-%d"
)
OUTPUT_FOLDER.mkdir(exist_ok=True, parents=True)

# Load configuration
DATASET_ROOT = Path("/workspace/data/datasets/NORDSTEN/bids")
INDEX_PATH = DATASET_ROOT / "index.csv"
REINDEX_DATASET = False

PRIMARY_QUERY = "`meta_MagneticFieldStrength` == 1.5 and ~is_sidecar and `source` == 'raw' and ~age_at_baseline.isna() and ~odi_baseline.isna() and additional_suffixes.isna() and `sub-cohort` != 'listhesis'"

REFERENCE_IMAGES_FOLDER = Path("/workspace/data/atlas/healthy/")

In [ ]:
#
# Load main dataset
#

if REINDEX_DATASET or not INDEX_PATH.exists():
    print("Reindexing dataset...")
    layout = Layout(root=DATASET_ROOT, include_derivatives=True)
    layout.index(load_sidecars=True)
    subject_metadata = pd.read_csv(DATASET_ROOT / "participants.tsv", sep="\t")
    layout.join_subject_metadata(subject_metadata, on="participant_id")
    layout.save_index(INDEX_PATH, ignore_encoding_errors=True)
else:
    layout = Layout(DATASET_ROOT)
    layout.load_index(INDEX_PATH)

In [ ]:
#
# Fill metadata from sidecars
#

df = layout._df.copy()

# Fill metadata
meta_cols = [column for column in df.columns if column.startswith("meta_")]
keys = ["sub", "acq", "suffix"]

df.update(df.groupby(keys)[meta_cols].transform(lambda g: g.ffill().bfill()))

# Method

## Images for pre-processing flowchart

In [ ]:
ref_image = load_image(Path("/workspace/data/figures/healthy-sub-0004_T2w.nii.gz"))
ref_label_map = load_label_map(
    Path("/workspace/data/figures/healthy-sub-0004_T2w_dseg.nii.gz")
)
ref_levels = load_label_map(
    Path("/workspace/data/figures/healthy-sub-0004_T2w_levels.nii.gz")
)

ref_processed_image = load_image(
    Path("/workspace/data/figures/healthy-sub-0004-straightened_T2w.nii.gz")
)
ref_processed_label_map = load_label_map(
    Path("/workspace/data/figures/healthy-sub-0004-straightened_T2w_dseg.nii.gz")
)
ref_processed_levels = load_label_map(
    Path("/workspace/data/figures/healthy-sub-0004-straightened_T2w_levels.nii.gz")
)
ref_processed_mask = load_label_map(
    Path("/workspace/data/figures/healthy-sub-0004-straightened_T2w_mask.nii.gz")
)

stenosis_image_entry = layout.query(
    "`sub` == '0013' and ~is_sidecar and suffix == 'T2w' and source == 'raw'"
)
stenosis_image = load_image(stenosis_image_entry["path"].item())
stenosis_label_map = load_label_map(
    layout.find_derivative(
        stenosis_image_entry,
        flt="pipeline == 'totalspineseg' and `additional_suffixes` == 'dseg'",
    )["path"]
)
stenosis_levels = load_label_map(
    layout.find_derivative(
        stenosis_image_entry,
        flt="pipeline == 'totalspineseg' and `additional_suffixes` == 'levels'",
    )["path"]
)
stenosis_resampled_image = load_image(
    layout.find_derivative(
        stenosis_image_entry,
        flt="pipeline == 'registered_straightened_single_image' and additional_suffixes.isna()",
    )["path"]
)
stenosis_resampled_mask = load_label_map(
    layout.find_derivative(
        stenosis_image_entry,
        flt="pipeline == 'registered_straightened_single_image' and `additional_suffixes` == 'mask'",
    )["path"]
)
stenosis_resampled_segmentation = load_label_map(
    layout.find_derivative(
        stenosis_image_entry,
        flt="pipeline == 'registered_straightened_single_image' and `additional_suffixes` == 'dseg'",
    )["path"]
)
stenosis_resampled_normalized_image = load_image(
    layout.find_derivative(
        stenosis_image_entry,
        flt="pipeline == 'registered_straightened_single_image_normalized'",
    )["path"]
)

In [ ]:
SLICE_INDEX = 8


def get_slice(image, slice_index=SLICE_INDEX):
    return sitk.GetArrayFromImage(image)[0:512, 0:512, slice_index]


def plot_image(ax, image, slice_index=SLICE_INDEX):
    image_slice = get_slice(image, slice_index)
    ax.imshow(image_slice, cmap="gray", origin="lower")
    ax.axis("equal")


def plot_label_map(ax, label_map, intervals, alpha=0.5, slice_index=SLICE_INDEX):
    label_map_arr = get_slice(label_map, slice_index)
    remapped = np.zeros_like(label_map_arr, dtype=int)
    colors = []

    for idx, (min_label, max_label, color) in enumerate(intervals, start=1):
        mask = (label_map_arr >= min_label) & (label_map_arr <= max_label)
        remapped[mask] = idx
        colors.append(color)

    masked = np.ma.masked_where(remapped == 0, remapped)
    cmap = ListedColormap(colors)
    ax.imshow(masked, cmap=cmap, alpha=alpha, vmin=1, vmax=len(colors), origin="lower")
    ax.axis("equal")


def plot_histogram(
    ax,
    image,
    mask,
    bins=50,
    sigma=2,
    slice_index=SLICE_INDEX,
    color=None,
    linestyle=None,
):
    mask_arr = get_slice(mask, slice_index)
    image_arr = get_slice(image, slice_index)
    data = image_arr[mask_arr != 0]

    counts, bin_edges = np.histogram(data, bins=bins)
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    smooth_counts = gaussian_filter1d(counts, sigma=sigma)

    ax.plot(centers, smooth_counts, linewidth=1.5, color=color, linestyle=linestyle)
    ax.fill_between(centers, smooth_counts, alpha=0.3, color=color)


def make_arrow_axes(ax):
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    x_min, x_max = ax.get_xlim()
    y_min, y_max = ax.get_ylim()

    ax.annotate(
        "",
        xy=(x_max, 0),
        xytext=(0, 0),
        arrowprops=dict(arrowstyle="->", lw=1.5),
        clip_on=False,
    )

    ax.annotate(
        "",
        xy=(0, y_max),
        xytext=(0, 0),
        arrowprops=dict(arrowstyle="->", lw=1.5),
        clip_on=False,
    )


ref_image_bundle = ImageBundle(
    image=ref_image,
    label_map=ref_label_map,
    levels=ref_levels,
)

stenosis_image_bundle = ImageBundle(
    image=stenosis_image, label_map=stenosis_label_map, levels=stenosis_levels
)

fig, axs = plt.subplots(2, 7, figsize=(12, 3.5), dpi=DPI)

for ax in np.ravel(axs):
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

# Segmentation
plot_image(axs[0, 0], stenosis_image)
plot_label_map(
    axs[0, 0],
    stenosis_label_map,
    [(2, 2, "tab:green"), (11, 50, "tab:orange"), (63, 100, "tab:blue")],
)

plot_image(axs[1, 0], ref_image)
plot_label_map(
    axs[1, 0],
    ref_label_map,
    [(2, 2, "tab:green"), (11, 50, "tab:orange"), (63, 100, "tab:blue")],
)

# Straightening
straightened_stenosis_image_bundle = stenosis_image_bundle.straightened()
plot_image(axs[0, 1], straightened_stenosis_image_bundle.image)

straightened_ref_image_bundle = (
    ref_image_bundle.straightened().with_harmonized_directions()
)
plot_image(axs[1, 1], straightened_ref_image_bundle.image)

# Masking
masked_stenosis_image_bundle = straightened_stenosis_image_bundle.with_roi_mask()
plot_image(axs[0, 2], masked_stenosis_image_bundle.image)
plot_label_map(axs[0, 2], masked_stenosis_image_bundle.mask, [(1, 1, "tab:blue")])

masked_ref_image_bundle = straightened_ref_image_bundle.with_roi_mask()
plot_image(axs[1, 2], masked_ref_image_bundle.image)
plot_label_map(axs[1, 2], masked_ref_image_bundle.mask, [(1, 1, "tab:blue")])

# Intensity normalization to CSF
plot_histogram(
    axs[0, 3],
    masked_stenosis_image_bundle.image,
    masked_stenosis_image_bundle.mask,
    color="tab:blue",
    linestyle="dashed",
)
plot_histogram(
    axs[0, 3],
    masked_stenosis_image_bundle.image / 2,
    masked_stenosis_image_bundle.mask,
    color="tab:blue",
    linestyle="solid",
)
make_arrow_axes(axs[0, 3])

axs[0, 3].annotate(
    "",
    xy=(0.5, 0.9),
    xytext=(0.95, 0.9),
    xycoords="axes fraction",
    textcoords="axes fraction",
    arrowprops=dict(arrowstyle="->", lw=2),
    annotation_clip=False,
)

plot_histogram(
    axs[1, 3],
    ref_processed_image,
    ref_processed_mask,
    color="tab:orange",
    linestyle="dashed",
)
plot_histogram(
    axs[1, 3],
    ref_processed_image / 2,
    ref_processed_mask,
    color="tab:orange",
    linestyle="solid",
)
make_arrow_axes(axs[1, 3])

axs[1, 3].annotate(
    "",
    xy=(0.5, 0.9),
    xytext=(0.95, 0.9),
    xycoords="axes fraction",
    textcoords="axes fraction",
    arrowprops=dict(arrowstyle="->", lw=2),
    annotation_clip=False,
)

# Registration
plot_image(
    axs[0, 4],
    sitk.CheckerBoard(
        sitk.Mask(stenosis_resampled_image, stenosis_resampled_mask),
        sitk.Mask(ref_processed_image, ref_processed_mask),
        (8, 8, 8),
    ),
)

# Intensity matching
plot_histogram(
    axs[0, 5],
    stenosis_resampled_normalized_image,
    stenosis_resampled_mask,
    color="tab:blue",
)
plot_histogram(axs[0, 5], ref_processed_image, ref_processed_mask, color="tab:orange")
make_arrow_axes(axs[0, 5])

# Done
plot_image(
    axs[0, 6],
    sitk.CheckerBoard(
        sitk.Mask(stenosis_resampled_normalized_image, stenosis_resampled_mask),
        sitk.Mask(ref_processed_image, ref_processed_mask),
        (8, 8, 8),
    ),
)

plt.show()

In [ ]:
if SAVE_FILES:
    output_file = (
        OUTPUT_FOLDER / "pre-processing-flowchart/pre-processing-flowchart.png"
    )
    output_file.parent.mkdir(exist_ok=True, parents=True)

    renderer = fig.canvas.get_renderer()

    for i, ax in enumerate(np.ravel(axs)):
        bbox = ax.get_tightbbox(renderer).transformed(fig.dpi_scale_trans.inverted())

        for suffix in IMAGE_FILE_FORMATS_TO_EXPORT:
            fig.savefig(
                output_file.with_name(f"{output_file.stem}_{i}{suffix}"),
                bbox_inches=bbox,
            )

# Result

## Subject characteristics 

Generate a table and preferably be reusable for both the primary analysis and stratified analysis

In [ ]:
df_primary = df.query(PRIMARY_QUERY)


def summarize(df):
    def median_iqr_str(series):
        median = series.median()
        q25 = series.quantile(0.25)
        q75 = series.quantile(0.75)
        return f"{median} [{q25}-{q75}]"

    df_unique_subjects = df[~df.duplicated(["sub"], keep="first")]

    summary = {}

    summary["number_of_subjects"] = len(df_unique_subjects)
    summary["ratio_female_percent"] = (
        (df_unique_subjects["sex"] == "F").sum() / len(df_unique_subjects) * 100
    )
    summary["age"] = median_iqr_str(df_unique_subjects["age_at_baseline"])
    summary["odi"] = median_iqr_str(df_unique_subjects["odi_baseline"])

    # Number of T1w and T2w images
    df_t1w = df[df["suffix"] == "T1w"]
    df_t2w = df[df["suffix"] == "T2w"]

    summary["number_t1w_images"] = len(df_t1w)
    summary["number_t2w_images"] = len(df_t2w)

    # Echo time stratified on T1w and T2w
    summary["echo_time_t1w"] = median_iqr_str(df_t1w["meta_EchoTime"])
    summary["echo_time_t2w"] = median_iqr_str(df_t2w["meta_EchoTime"])

    # Repeition time stratified on T1w and T2w
    summary["repetition_time_t1w"] = median_iqr_str(df_t1w["meta_RepetitionTime"])
    summary["repetition_time_t2w"] = median_iqr_str(df_t2w["meta_RepetitionTime"])

    return summary


pd.DataFrame(
    {
        "SST": summarize(df_primary[df_primary["cohort"] == "SST"]),
        "OC": summarize(df_primary[df_primary["cohort"] == "OC"]),
        "overall": summarize(df_primary),
    }
)

## Stratified Subject characteristics 

In [ ]:
# Add check on how well the narrowest level could be detemined when comparing to radiologicsts.
pd.DataFrame(
    {
        "L2-L3 narrowest": summarize(
            df_primary[
                df_primary["narrowest_isolated_level_from_segmentation_at_baseline"]
                == "L2-L3"
            ]
        ),
        "L3-L4 narrowest": summarize(
            df_primary[
                df_primary["narrowest_isolated_level_from_segmentation_at_baseline"]
                == "L3-L4"
            ]
        ),
        "L4-L5 narrowest": summarize(
            df_primary[
                df_primary["narrowest_isolated_level_from_segmentation_at_baseline"]
                == "L4-L5"
            ]
        ),
    }
)

In [ ]:
df_not_nan_narrowest_level = df_primary[
    df_primary["narrowest_level_at_baseline"].notna()
    & df_primary["narrowest_level_from_segmentation_at_baseline"].notna()
]
sum_narrowest_level_same = (
    df_not_nan_narrowest_level["narrowest_level_at_baseline"]
    == df_not_nan_narrowest_level["narrowest_level_from_segmentation_at_baseline"]
).sum()

print(
    f"Agreement between radiological deteminerd narrowest level and assesment from segmentation: {sum_narrowest_level_same / len(df_not_nan_narrowest_level) * 100} %"
)

## DICE score table

Generate a table listning the DICE scores for the registration.

In [ ]:
df_dice = df_primary[
    ["sub", "suffix"]
    + [column for column in df_primary.columns if "meta_Dice" in column]
].melt(id_vars=["sub", "suffix"], var_name="label")


def get_label(x):
    label = x.split("_")[2]
    if str.isnumeric(label):
        return TotalSpineSegLabels(int(label)).name.replace("_", " ").title()
    else:
        return label.title()


df_dice["label"] = df_dice["label"].apply(get_label)


def median_iqr_latex(x):
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    med = x.median()
    return f"{med:.3f} ({q1:.3f}--{q3:.3f})"


dice_table = (
    df_dice.groupby(["label", "suffix"])["value"].apply(median_iqr_latex).unstack()
)

dice_table

## Set of reference images

In [ ]:
def load_reference_image(folder, suffix, load_straightened):
    path = [
        path
        for path in folder.glob(f"*{suffix}.nii.gz")
        if ("straightened" in path.name) == load_straightened
    ][0]
    print(f"Loading image: {path}")
    image = load_image(path)
    return sitk.GetArrayFromImage(image)


t1w_image_ref = load_reference_image(REFERENCE_IMAGES_FOLDER, "T1w", False)
t1w_image_straightened_ref = load_reference_image(REFERENCE_IMAGES_FOLDER, "T1w", True)
t2w_image_ref = load_reference_image(REFERENCE_IMAGES_FOLDER, "T2w", False)
t2w_image_straightened_ref = load_reference_image(REFERENCE_IMAGES_FOLDER, "T2w", True)

In [ ]:
def plot_image(ax, image, key):
    ax.imshow(image, cmap="gray", origin="lower")
    ax.text(
        0.05,
        0.95,
        key,
        ha="left",
        va="top",
        color="white",
        fontweight="bold",
        transform=ax.transAxes,
        size=12,
    )
    ax.axis("off")


fig, axs = plt.subplots(1, 4, layout="constrained", figsize=(4, 1), dpi=DPI)

axs_flat = np.ravel(axs)

plot_image(axs_flat[0], t1w_image_ref[:, :, 8], "a")
plot_image(axs_flat[1], t1w_image_straightened_ref[0:256, 0:256, 8], "b")
plot_image(axs_flat[2], t2w_image_ref[:, :, 8], "c")
plot_image(axs_flat[3], t2w_image_straightened_ref[0:512, 0:512, 8], "d")

plt.show()

In [ ]:
if SAVE_FILES:
    output_file = OUTPUT_FOLDER / "reference_images/reference_images.png"
    output_file.parent.mkdir(exist_ok=True, parents=True)
    for suffix in IMAGE_FILE_FORMATS_TO_EXPORT:
        fig.savefig(output_file.with_suffix(suffix))

## Intensity normalization

In [ ]:
def get_manufacturer_from_str(manufacturer_string):
    s = str(manufacturer_string).upper()
    if "SIEMENS" in s:
        return "SIEMENS"
    elif "PHILIPS" in s:
        return "PHILIPS"
    elif "GE" in s:
        return "GE"
    elif "TOSHIBA" in s:
        return "TOSHIBA"
    return "OTHER"


tissue_label_ranges = {
    "Discs": (63, 100),
    "Vertebrae": (11, 45),
}

num_bins = 512
num_interpolated_bins = 128

hist_data = defaultdict(
    lambda: defaultdict(
        lambda: defaultdict(
            lambda: {
                "raw": {"histograms": [], "bin_edges": []},
                "scaled": {"histograms": [], "bin_edges": []},
                "norm": {"histograms": [], "bin_edges": []},
            }
        )
    )
)

for weight in ["T1w", "T2w"]:
    for count, (index, entry) in enumerate(
        df_primary[df_primary["suffix"] == weight].iterrows(), start=1
    ):
        manufacturer = get_manufacturer_from_str(entry["meta_Manufacturer"])

        image_raw_path = entry["path"]
        image_raw_label_map_path = layout.find_derivative(
            entry,
            flt="pipeline == 'totalspineseg' and  `additional_suffixes` == 'dseg'",
        )["path"]

        image_scaled_path = layout.find_derivative(
            entry,
            flt="pipeline == 'registered_straightened_single_image' and additional_suffixes.isna()",
        )["path"]
        image_scaled_label_map_path = layout.find_derivative(
            entry,
            flt="pipeline == 'registered_straightened_single_image' and `additional_suffixes` == 'dseg'",
        )["path"]

        image_norm_path = layout.find_derivative(
            entry,
            flt="pipeline == 'registered_straightened_single_image_normalized' and additional_suffixes.isna()",
        )["path"]

        image_raw_arr = sitk.GetArrayFromImage(load_image(image_raw_path))
        image_raw_label_map_arr = sitk.GetArrayFromImage(
            load_label_map(image_raw_label_map_path)
        )

        image_scaled_arr = sitk.GetArrayFromImage(load_image(image_scaled_path))
        image_scaled_label_map_arr = sitk.GetArrayFromImage(
            load_label_map(image_scaled_label_map_path)
        )

        image_norm_arr = sitk.GetArrayFromImage(load_image(image_norm_path))
        image_norm_label_map_arr = image_scaled_label_map_arr

        for tissue_name, (label_min, label_max) in tissue_label_ranges.items():
            raw_values = image_raw_arr[
                (image_raw_label_map_arr >= label_min)
                & (image_raw_label_map_arr <= label_max)
            ].ravel()
            scaled_values = image_scaled_arr[
                (image_scaled_label_map_arr >= label_min)
                & (image_scaled_label_map_arr <= label_max)
            ].ravel()
            norm_values = image_norm_arr[
                (image_norm_label_map_arr >= label_min)
                & (image_norm_label_map_arr <= label_max)
            ].ravel()

            hist_raw, bins_raw = np.histogram(raw_values, bins=num_bins, density=True)
            hist_scaled, bins_scaled = np.histogram(
                scaled_values, bins=num_bins, density=True
            )
            hist_norm, bins_norm = np.histogram(
                norm_values, bins=num_bins, density=True
            )

            hist_data[weight][tissue_name][manufacturer]["raw"]["histograms"].append(
                hist_raw
            )
            hist_data[weight][tissue_name][manufacturer]["raw"]["bin_edges"].append(
                bins_raw
            )

            hist_data[weight][tissue_name][manufacturer]["scaled"]["histograms"].append(
                hist_scaled
            )
            hist_data[weight][tissue_name][manufacturer]["scaled"]["bin_edges"].append(
                bins_scaled
            )

            hist_data[weight][tissue_name][manufacturer]["norm"]["histograms"].append(
                hist_norm
            )
            hist_data[weight][tissue_name][manufacturer]["norm"]["bin_edges"].append(
                bins_norm
            )

In [ ]:
def interpolate_histograms_to_common_grid(histograms, bin_edges_list, num_bins):
    bin_edges_arr = np.asarray(bin_edges_list)
    upper = np.percentile(bin_edges_arr.ravel(), 80)
    global_bins = np.linspace(0, upper, num_bins)

    interpolated = []

    for hist, bin_edges in zip(histograms, bin_edges_list):
        if (
            hist is None
            or np.any(np.isnan(hist))
            or bin_edges is None
            or np.any(np.isnan(bin_edges))
        ):
            continue

        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        interp_hist = np.interp(
            global_bins,
            bin_centers,
            hist,
            left=0.0,
            right=0.0,
        )
        interpolated.append(interp_hist)

    return np.asarray(interpolated), global_bins


styles = [
    {
        "name": "Vendor 1",
        "color": "#648FFF",
        "linestyle": "-",
        "hatch": "///",
    },
    {
        "name": "Vendor 2",
        "color": "#785EF0",
        "linestyle": "--",
        "hatch": "\\\\",
    },
    {
        "name": "Vendor 3",
        "color": "#DC267F",
        "linestyle": "-.",
        "hatch": "xxx",
    },
    {"name": "Vendor 4", "color": "#FE6100", "linestyle": ":", "hatch": "..."},
]

styles_for_vendor = {
    manufacturer: styles[i]
    for i, manufacturer in enumerate(
        hist_data["T1w"][list(tissue_label_ranges.keys())[0]].keys()
    )
}

n_tissues = len(tissue_label_ranges)

fig, axs = plt.subplots(n_tissues * 2, 3, figsize=(2.5 * 3, 2 * n_tissues * 2))

for ax in axs[-1, :]:
    ax.set_xlabel("Signal [arb. unit.]")

for ax in axs[:, 0]:
    ax.set_ylabel("Density", fontsize=12)

for weight_idx, weight in enumerate(["T1w", "T2w"]):
    for tissue_idx, tissue_name in enumerate(tissue_label_ranges.keys()):
        ax_raw = axs[weight_idx + tissue_idx * 2, 0]
        ax_scaled = axs[weight_idx + tissue_idx * 2, 1]
        ax_norm = axs[weight_idx + tissue_idx * 2, 2]

        for manufacturer, manufacturer_data in hist_data[weight][tissue_name].items():
            style = styles_for_vendor.get(manufacturer)

            # Raw
            raw_histograms = manufacturer_data["raw"]["histograms"]
            raw_bin_edges = manufacturer_data["raw"]["bin_edges"]

            raw_interp, global_bins = interpolate_histograms_to_common_grid(
                raw_histograms,
                raw_bin_edges,
                num_interpolated_bins,
            )

            q25 = np.percentile(raw_interp, 25, axis=0)
            median = np.median(raw_interp, axis=0)
            q75 = np.percentile(raw_interp, 75, axis=0)

            ax_raw.plot(
                global_bins,
                median,
                label=style["name"],
                color=style["color"],
                linestyle=style["linestyle"],
            )
            ax_raw.fill_between(
                global_bins,
                q25,
                q75,
                alpha=0.2,
                color=style["color"],
                hatch=style["hatch"],
                edgecolor=style["color"],
            )

            # Scaled
            scaled_histograms = manufacturer_data["scaled"]["histograms"]
            scaled_bin_edges = manufacturer_data["scaled"]["bin_edges"]

            scaled_interp, global_bins = interpolate_histograms_to_common_grid(
                scaled_histograms,
                scaled_bin_edges,
                num_interpolated_bins,
            )

            q25 = np.percentile(scaled_interp, 25, axis=0)
            median = np.median(scaled_interp, axis=0)
            q75 = np.percentile(scaled_interp, 75, axis=0)

            ax_scaled.plot(
                global_bins,
                median,
                label=style["name"],
                color=style["color"],
                linestyle=style["linestyle"],
            )
            ax_scaled.fill_between(
                global_bins,
                q25,
                q75,
                alpha=0.2,
                color=style["color"],
                hatch=style["hatch"],
                edgecolor=style["color"],
            )

            # Normalized
            norm_histograms = manufacturer_data["norm"]["histograms"]
            norm_bin_edges = manufacturer_data["norm"]["bin_edges"]

            norm_interp, global_bins = interpolate_histograms_to_common_grid(
                norm_histograms,
                norm_bin_edges,
                num_interpolated_bins,
            )
            if norm_interp is not None:
                q25 = np.percentile(norm_interp, 25, axis=0)
                median = np.median(norm_interp, axis=0)
                q75 = np.percentile(norm_interp, 75, axis=0)

                ax_norm.plot(
                    global_bins,
                    median,
                    label=style["name"],
                    color=style["color"],
                    linestyle=style["linestyle"],
                )
                ax_norm.fill_between(
                    global_bins,
                    q25,
                    q75,
                    alpha=0.2,
                    color=style["color"],
                    hatch=style["hatch"],
                    edgecolor=style["color"],
                )

        ax_raw.grid(True)
        ax_scaled.grid(True)
        ax_norm.grid(True)

        ax_raw.set_title(tissue_name, loc="left")
        ax_raw.set_title(f"{weight} (raw)", loc="right")

        ax_scaled.set_title(tissue_name, loc="left")
        ax_scaled.set_title(f"{weight} (scaled)", loc="right")

        ax_norm.set_title(tissue_name, loc="left")
        ax_norm.set_title(f"{weight} (norm)", loc="right")

handles, labels = axs.flat[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    frameon=False,
    ncols=len(handles),
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
)

fig.tight_layout(rect=[0.0, 0.09, 1.00, 1], pad=0.1)

plt.show()

In [ ]:
if SAVE_FILES:
    output_file = (
        OUTPUT_FOLDER
        / "intensity-normalization-result/intensity-normalization-result.png"
    )
    output_file.parent.mkdir(exist_ok=True, parents=True)
    for suffix in IMAGE_FILE_FORMATS_TO_EXPORT:
        fig.savefig(output_file.with_suffix(suffix))

## GLM analysis

In [ ]:
reference_image_dir = "/workspace/data/atlas/healthy/"

image_file_formats_to_export = [".eps", ".tiff", ".png", ".pdf"]

common_plot_config = {
    "weights": {
        "T1w": {"slices": [8, 12], "roi": ((20, 20), (220, 240))},
        "T2w": {
            "slices": [8, 12],
            "roi": ((40, 40), (440, 480)),
        },
    },
    "regressors": {
        "narrowest_relative_widest_estimated_dsca_T2w": "Spinal canal area",
        "age_at_baseline": "Age",
        "sex": "Female gender",
        "odi_baseline": "ODI",
    },
    "p_value": 0.05,
    "neg_log10_p_value": "logp_max_tfce",
    "cmap": Colormap("crameri:vik").to_matplotlib(),
}

plot_config_effect_size = {
    "contrast": "effect_size",
    "contrast_description": "Effect size",
    "norm": TwoSlopeNorm(0, vmin=-0.5, vmax=0.5),
} | common_plot_config

plot_config_z_score = {
    "contrast": "z_score",
    "contrast_description": "Z-Score",
    "norm": TwoSlopeNorm(0, vmin=-10.0, vmax=10.0),
} | common_plot_config

In [ ]:
def resolve_path_first(path) -> Path:
    folder, pattern = path.rsplit("/", maxsplit=1)
    matches = list(Path(folder).glob(pattern))

    if len(matches) != 1:
        raise ValueError("Pattern didn't match exactly one file.")

    return matches[0]


def get_available_regressors(result_dir, plot_config):
    return {
        k: v
        for k, v in plot_config["regressors"].items()
        if len(list(Path(result_dir).glob(f"{k}_*"))) != 0
    }


def plot_glm_result(plot_config, result_dir, reference_image_dir, dpi):
    cols = len(
        plot_config["weights"][next(iter(plot_config["weights"]))]["slices"]
    ) * len(plot_config["weights"])

    valid_regressors = get_available_regressors(result_dir, plot_config)

    rows = len(valid_regressors)

    fig = plt.figure(figsize=(130 / 25.4, 120 / 25.4), dpi=dpi)

    gs0 = gridspec.GridSpec(2, 1, figure=fig, height_ratios=[0.975, 0.025])

    gs1 = gridspec.GridSpecFromSubplotSpec(rows, cols, subplot_spec=gs0[0])
    axs = [
        fig.add_subplot(gs1[row, col]) for row, col in product(range(rows), range(cols))
    ]

    gs2 = gridspec.GridSpecFromSubplotSpec(
        1, 2, subplot_spec=gs0[1], width_ratios=[1.4, 1.0], wspace=0.02
    )
    ax_colorbar_label = fig.add_subplot(gs2[0])
    ax_colorbar = fig.add_subplot(gs2[1])

    axis_index = 0

    for regressor, weight in product(
        valid_regressors.keys(), plot_config["weights"].keys()
    ):
        image_arr = sitk.GetArrayFromImage(
            load_image(
                resolve_path_first(
                    f"{reference_image_dir}*straightened_{weight}.nii.gz"
                )
            )
        )

        negative_log_pval_arr = sitk.GetArrayFromImage(
            load_image(
                resolve_path_first(
                    f"{result_dir}/{regressor}_{plot_config['neg_log10_p_value']}_{weight}.nii.gz"
                )
            )
        )

        contrast_arr = sitk.GetArrayFromImage(
            load_image(
                resolve_path_first(
                    f"{result_dir}/{regressor}_{plot_config['contrast']}_{weight}.nii.gz"
                )
            )
        )

        contrast_arr[negative_log_pval_arr < -np.log10(plot_config["p_value"])] = np.nan

        for slice_index in plot_config["weights"][weight]["slices"]:
            ax = axs[axis_index]

            ax.imshow(image_arr[:, :, slice_index], cmap="gray")

            cbar_ref = ax.imshow(
                contrast_arr[:, :, slice_index],
                cmap=plot_config["cmap"],
                norm=plot_config["norm"],
            )

            ax.text(
                0.01,
                0.99,
                string.ascii_lowercase[axis_index],
                color="white",
                fontsize=12,
                fontweight="bold",
                transform=ax.transAxes,
                va="top",
            )

            ax.text(
                0.01,
                0.01,
                f"slice {slice_index}",
                color="white",
                fontsize=12,
                transform=ax.transAxes,
                va="bottom",
                ha="left",
            )

            ax.tick_params(
                axis="both",
                which="both",
                bottom=False,
                top=False,
                left=False,
                right=False,
                labelbottom=False,
                labelleft=False,
            )

            roi = plot_config["weights"][weight]["roi"]

            ax.set_xlim(roi[0][0], roi[1][0])
            ax.set_ylim(roi[0][1], roi[1][1])

            ax.set_aspect("equal", adjustable="box")
            ax.margins(0)

            # Set title for first rows
            if axis_index < cols:
                ax.set_title(weight, fontsize=12)

            # Set regressor name
            if axis_index % cols == 0:
                ax.set_ylabel(plot_config["regressors"][regressor], fontsize=12)

            axis_index = axis_index + 1

    ax_colorbar_label.axis("off")
    ax_colorbar_label.text(
        0,
        0.5,
        f"{plot_config['contrast_description']} masked to p < 0.05",
        fontsize=12,
        va="center",
        ha="left",
    )
    cbar = fig.colorbar(
        cbar_ref, cax=ax_colorbar, orientation="horizontal", extend="both"
    )
    cbar.ax.tick_params(labelsize=12)

    fig.subplots_adjust(wspace=0.02, hspace=0.02)
    fig.tight_layout()

    return fig


def plot_design_matrix_correlation(plot_config, result_dir, reference_image_dir, dpi):
    corr_max = None
    valid_regressors = get_available_regressors(result_dir, plot_config)

    # Calculate correlation matrices
    for weight in plot_config["weights"].keys():
        design_matrix_path = Path(f"{result_dir}/design_matrix_{weight}.csv")

        design_matrix = pd.read_csv(design_matrix_path)
        design_matrix_regressors = design_matrix[valid_regressors.keys()]

        corr = design_matrix_regressors.corr(method="pearson")

        if corr_max is None:
            corr_max = corr
        else:
            # Ensure same ordering
            corr = corr.loc[corr_max.index, corr_max.columns]
            # Find max correlations
            corr_max = pd.DataFrame(
                np.where(
                    np.abs(corr.values) >= np.abs(corr_max.values),
                    corr.values,
                    corr_max.values,
                ),
                index=corr_max.index,
                columns=corr_max.columns,
            )

    num_regressors = len(valid_regressors)

    # Plot
    fig, ax = plt.subplots(figsize=(80 / 25.4, 60 / 25.4), dpi=dpi)

    im = ax.imshow(
        corr_max,
        cmap=Colormap("crameri:vik").to_matplotlib(),
        norm=TwoSlopeNorm(0, vmin=-1, vmax=1),
    )

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.1)
    fig.colorbar(im, cax=cax)

    ax.set_xticks(np.arange(num_regressors))
    ax.set_yticks(np.arange(num_regressors))

    regressor_names = [name.replace(" ", "\n") for name in valid_regressors.values()]

    ax.set_xticklabels(regressor_names, rotation=45, ha="right", fontsize=12)
    ax.set_yticklabels(regressor_names, fontsize=12)

    # Annotate values inside the matrix
    for i in range(num_regressors):
        for j in range(num_regressors):
            ax.text(
                j,
                i,
                f"{corr_max.iloc[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=12,
            )

    fig.tight_layout()

    return fig

### Main voxel-wise analysis with normalization

In [ ]:
result_dir = "/workspace/data/results/glm/sst-oc-baseline/"

fig_z_score = plot_glm_result(
    plot_config_z_score, result_dir, reference_image_dir, dpi=DPI
)
fig_effect_size = plot_glm_result(
    plot_config_effect_size, result_dir, reference_image_dir, dpi=DPI
)
fig_correlation_matrix = plot_design_matrix_correlation(
    plot_config_effect_size, result_dir, reference_image_dir, dpi=DPI
)

if SAVE_FILES:
    z_score_output_path = (
        OUTPUT_FOLDER
        / f"glm-sst-oc/glm-sst-oc_p-value-{common_plot_config['p_value']}_z_score.png"
    )
    z_score_output_path.parent.mkdir(exist_ok=True, parents=True)
    effect_size_output_path = (
        OUTPUT_FOLDER
        / f"glm-sst-oc/glm-sst-oc_p-value-{common_plot_config['p_value']}_effect_size.png"
    )
    effect_size_output_path.parent.mkdir(exist_ok=True, parents=True)
    correlation_matrix_output_path = (
        OUTPUT_FOLDER / "glm-sst-oc/glm-sst-oc_correlation-matrix.png"
    )
    for ext in image_file_formats_to_export:
        fig_z_score.savefig(z_score_output_path.with_suffix(ext))
        fig_effect_size.savefig(effect_size_output_path.with_suffix(ext))
        fig_correlation_matrix.savefig(correlation_matrix_output_path.with_suffix(ext))

### Main analysis without histogram based normalization

In [ ]:
result_dir_only_csf_normalization = (
    "/workspace/data/results/glm/sst-oc-baseline-only-csf-normalization/"
)

fig_z_score_only_csf_normalization = plot_glm_result(
    plot_config_z_score, result_dir_only_csf_normalization, reference_image_dir, dpi=DPI
)
fig_effect_size_only_csf_normalization = plot_glm_result(
    plot_config_effect_size,
    result_dir_only_csf_normalization,
    reference_image_dir,
    dpi=DPI,
)

if SAVE_FILES:
    z_score_output_path = (
        OUTPUT_FOLDER
        / f"glm-sst-oc-only-csf-normalization/glm-sst-oc-only-csf-normalization_p-value-{common_plot_config['p_value']}_z_score.png"
    )
    z_score_output_path.parent.mkdir(exist_ok=True, parents=True)
    effect_size_output_path = (
        OUTPUT_FOLDER
        / f"glm-sst-oc-only-csf-normalization/glm-sst-oc-only-csf-normalization_p-value-{common_plot_config['p_value']}_effect_size.png"
    )
    effect_size_output_path.parent.mkdir(exist_ok=True, parents=True)
    for ext in image_file_formats_to_export:
        fig_z_score_only_csf_normalization.savefig(z_score_output_path.with_suffix(ext))
        fig_effect_size_only_csf_normalization.savefig(
            effect_size_output_path.with_suffix(ext)
        )

### Stratified analysis on isolated narrowest level L4-L5 with normalization

In [ ]:
result_dir_l4_l5_isolated_narrowest = (
    "/workspace/data/results/glm/sst-L4-L5-isolated-narrowest/"
)

fig_z_score_isolated_narrowest = plot_glm_result(
    plot_config_z_score,
    result_dir_l4_l5_isolated_narrowest,
    reference_image_dir,
    dpi=DPI,
)
fig_effect_size_isolated_narrowest = plot_glm_result(
    plot_config_effect_size,
    result_dir_l4_l5_isolated_narrowest,
    reference_image_dir,
    dpi=DPI,
)
fig_correlation_matrix_isolated_narrowest = plot_design_matrix_correlation(
    plot_config_effect_size,
    result_dir_l4_l5_isolated_narrowest,
    reference_image_dir,
    dpi=DPI,
)

if SAVE_FILES:
    z_score_output_path = (
        OUTPUT_FOLDER
        / f"glm-isolated-narrowest-l4-l5/glm-isolated-narrowest-l4-l5_p-value-{common_plot_config['p_value']}_z_score.png"
    )
    z_score_output_path.parent.mkdir(exist_ok=True, parents=True)
    effect_size_output_path = (
        OUTPUT_FOLDER
        / f"glm-isolated-narrowest-l4-l5/glm-isolated-narrowest-l4-l5_p-value-{common_plot_config['p_value']}_effect_size.png"
    )
    effect_size_output_path.parent.mkdir(exist_ok=True, parents=True)
    correlation_matrix_output_path = (
        OUTPUT_FOLDER
        / "glm-isolated-narrowest-l4-l5/glm-isolated-narrowest-l4-l5_correlation-matrix.png"
    )
    for ext in image_file_formats_to_export:
        fig_z_score_isolated_narrowest.savefig(z_score_output_path.with_suffix(ext))
        fig_effect_size_isolated_narrowest.savefig(
            effect_size_output_path.with_suffix(ext)
        )
        fig_correlation_matrix_isolated_narrowest.savefig(
            correlation_matrix_output_path.with_suffix(ext)
        )

### Stratified analysis on isolated narrowest level L4-L5 without histogram based normalization

In [ ]:
result_dir_l4_l5_isolated_narrowest_only_csf_normalization = (
    "/workspace/data/results/glm/sst-only-csf-normalization-L4-L5-isolated-narrowest/"
)

fig_z_score_isolated_narrowest_only_csf_norm = plot_glm_result(
    plot_config_z_score,
    result_dir_l4_l5_isolated_narrowest_only_csf_normalization,
    reference_image_dir,
    dpi=DPI,
)
fig_effect_size_isolated_narrowest_only_csf_norm = plot_glm_result(
    plot_config_effect_size,
    result_dir_l4_l5_isolated_narrowest_only_csf_normalization,
    reference_image_dir,
    dpi=DPI,
)

if SAVE_FILES:
    z_score_output_path = (
        OUTPUT_FOLDER
        / f"glm-isolated-narrowest-l4-l5-only-csf-normalization/glm-isolated-narrowest-l4-l5-only-csf-normalization_p-value-{common_plot_config['p_value']}_z_score.png"
    )
    z_score_output_path.parent.mkdir(exist_ok=True, parents=True)
    effect_size_output_path = (
        OUTPUT_FOLDER
        / f"glm-isolated-narrowest-l4-l5-only-csf-normalization/glm-isolated-narrowest-l4-l5-only-csf-normalization_p-value-{common_plot_config['p_value']}_effect_size.png"
    )
    effect_size_output_path.parent.mkdir(exist_ok=True, parents=True)
    for ext in image_file_formats_to_export:
        fig_z_score_isolated_narrowest_only_csf_norm.savefig(
            z_score_output_path.with_suffix(ext)
        )
        fig_effect_size_isolated_narrowest_only_csf_norm.savefig(
            effect_size_output_path.with_suffix(ext)
        )

### Narrowing of the spinal canal as a predictor with histogram based normalization

In [ ]:
result_dir_narrowest_relative_widest_estimated_dcsa = (
    "/workspace/data/results/glm/narrowest_relative_widest_estimated_dcsa"
)

fig_z_score_rel_min_dcsa = plot_glm_result(
    plot_config_z_score,
    result_dir_narrowest_relative_widest_estimated_dcsa,
    reference_image_dir,
    dpi=DPI,
)
fig_effect_size_rel_min_dcsa = plot_glm_result(
    plot_config_effect_size,
    result_dir_narrowest_relative_widest_estimated_dcsa,
    reference_image_dir,
    dpi=DPI,
)
fig_correlation_matrix_rel_min_dcsa = plot_design_matrix_correlation(
    plot_config_effect_size,
    result_dir_narrowest_relative_widest_estimated_dcsa,
    reference_image_dir,
    dpi=DPI,
)

if SAVE_FILES:
    z_score_output_path = (
        OUTPUT_FOLDER
        / f"glm-narrowest-relative-widest-estimated-dcsa/glm-narrowest-relative-widest-estimated-dcsa_p-value-{common_plot_config['p_value']}_z_score.png"
    )
    z_score_output_path.parent.mkdir(exist_ok=True, parents=True)
    effect_size_output_path = (
        OUTPUT_FOLDER
        / f"glm-narrowest-relative-widest-estimated-dcsa/glm-narrowest-relative-widest-estimated-dcsa_p-value-{common_plot_config['p_value']}_effect_size.png"
    )
    effect_size_output_path.parent.mkdir(exist_ok=True, parents=True)
    correlation_matrix_output_path = (
        OUTPUT_FOLDER
        / "glm-narrowest-relative-widest-estimated-dcsa/glm-narrowest-relative-widest-estimated-dcsa_correlation-matrix.png"
    )
    for ext in image_file_formats_to_export:
        fig_z_score_rel_min_dcsa.savefig(z_score_output_path.with_suffix(ext))
        fig_effect_size_rel_min_dcsa.savefig(effect_size_output_path.with_suffix(ext))
        fig_correlation_matrix_rel_min_dcsa.savefig(
            correlation_matrix_output_path.with_suffix(ext)
        )

### Narrowing of the spinal canal as a predictor without histogram based normalization

In [ ]:
result_dir_narrowest_relative_widest_estimated_dcsa_only_csf_normalization = "/workspace/data/results/glm/narrowest_relative_widest_estimated_dcsa_only_csf_normalization"

fig_z_score_rel_min_dcsa = plot_glm_result(
    plot_config_z_score,
    result_dir_narrowest_relative_widest_estimated_dcsa_only_csf_normalization,
    reference_image_dir,
    dpi=DPI,
)
fig_effect_size_rel_min_dcsa = plot_glm_result(
    plot_config_effect_size,
    result_dir_narrowest_relative_widest_estimated_dcsa_only_csf_normalization,
    reference_image_dir,
    dpi=DPI,
)
fig_correlation_matrix_rel_min_dcsa = plot_design_matrix_correlation(
    plot_config_effect_size,
    result_dir_narrowest_relative_widest_estimated_dcsa_only_csf_normalization,
    reference_image_dir,
    dpi=DPI,
)

if SAVE_FILES:
    z_score_output_path = (
        OUTPUT_FOLDER
        / f"glm-narrowest-relative-widest-estimated-dcsa-only-csf-normalization/glm-narrowest-relative-widest-estimated-dcsa-only-csf-normalization_p-value-{common_plot_config['p_value']}_z_score.png"
    )
    z_score_output_path.parent.mkdir(exist_ok=True, parents=True)
    effect_size_output_path = (
        OUTPUT_FOLDER
        / f"glm-narrowest-relative-widest-estimated-dcsa-only-csf-normalization/glm-narrowest-relative-widest-estimated-dcsa-only-csf-normalization_p-value-{common_plot_config['p_value']}_effect_size.png"
    )
    effect_size_output_path.parent.mkdir(exist_ok=True, parents=True)
    correlation_matrix_output_path = (
        OUTPUT_FOLDER
        / "glm-narrowest-relative-widest-estimated-dcsa-only-csf-normalization/glm-narrowest-relative-widest-estimated-dcsa-only-csf-normalization_correlation-matrix.png"
    )
    for ext in image_file_formats_to_export:
        fig_z_score_rel_min_dcsa.savefig(z_score_output_path.with_suffix(ext))
        fig_effect_size_rel_min_dcsa.savefig(effect_size_output_path.with_suffix(ext))
        fig_correlation_matrix_rel_min_dcsa.savefig(
            correlation_matrix_output_path.with_suffix(ext)
        )